# Notebook Purpose

This notebook trains a YOLOv8 classification model to recognize body pose categories from full-body images of stand-up comedians, and then uses the trained model to classify a new set of images.

## Input:
- A CSV annotation file exported from Label Studio  
- A folder containing the corresponding annotated images (e.g., full-body screenshots)  
- A separate folder of new images to be classified by the model

## Processing steps:
1. Format the annotated dataset into the folder structure required by YOLOv8 (one folder per class)
2. Train a YOLOv8 model in classification mode using the annotated images
3. Apply the trained model to a new batch of images
4. Export the classification results to a CSV file

## Output:
- A trained YOLOv8 classification model (`.pt` file)
- A CSV file listing each classified image and its predicted pose category



In [ ]:
!pip install scikit-learn
#if needed 

## Step 1: Preparing the Dataset for YOLO Training



In [ ]:
import os
import shutil
import pandas as pd
import re
from sklearn.model_selection import train_test_split

In [ ]:
csv_file_path = 
images_dir = 
output_dir = 

# create output directories if they don't exist
train_dir = os.path.join(output_dir, 'train')
val_dir = os.path.join(output_dir, 'val')
test_dir = os.path.join(output_dir, 'test')
for dir_path in [train_dir, val_dir, test_dir]:
    os.makedirs(dir_path, exist_ok=True)

# load the annotation CSV file
annotations = pd.read_csv(csv_file_path)

#clean the image filename:
# 1. Remove the prefix "/data/upload/5/"
# 2. Remove the UUID (hexadecimal string followed by "_") at the beginning of the filename
annotations['image_clean'] = annotations['image'].apply(lambda x: re.sub(r'^[0-9a-fA-F]+_', '', os.path.basename(x)))

# remove classes with fewer than 2 images
class_counts = annotations["choice"].value_counts()
valid_classes = class_counts[class_counts >= 2].index  # Keep only classes with 2+ images
annotations = annotations[annotations["choice"].isin(valid_classes)]

# split the data into training, validation, and test sets with stratification
train_val, test = train_test_split(
    annotations, 
    test_size=0.15, 
    random_state=42, 
    stratify=annotations["choice"]  # Ensures balanced class distribution
)

train, val = train_test_split(
    train_val, 
    test_size=0.1765, 
    random_state=42, 
    stratify=train_val["choice"]  # Stratify again
)

# function to copy images into the appropriate directories
def copy_images(data, split):
    for _, row in data.iterrows():
        image_filename = row['image_clean']
        choice = row['choice']
        
        # check if the choice is a valid value (not NaN, not empty)
        if pd.notna(choice) and choice != '':
            label = str(choice)  # Convert to string
            
            # build full path to the original image
            original_image_path = os.path.join(images_dir, image_filename)
            
            # check if the file exists
            if not os.path.exists(original_image_path):
                print(f"⚠️ Image file not found: {original_image_path}")
                continue
            
            # create the class folder if it doesn't exist
            class_dir = os.path.join(output_dir, split, label)
            os.makedirs(class_dir, exist_ok=True)
            
            # copy the image to the class folder
            dst_path = os.path.join(class_dir, image_filename)
            shutil.copy(original_image_path, dst_path)

# copy images for each split
copy_images(train, 'train')
copy_images(val, 'val')
copy_images(test, 'test')

print("Done !")


## Step 2 : Training

In [ ]:
from ultralytics import YOLO


model = YOLO("yolov8s-cls.pt")  # load a pretrained model (recommended for training)

dataset_custom = "plans_extraits_pour_classifieur_poses_annotés_posesII"

# train
results = model.train(data=dataset_custom, epochs=100)


In [ ]:
#results are in  train 

## Step 3 :Validate the model

In [ ]:
# validate with defaults parameters
metrics = model.val()

## Step 4  : Apply the model

In [ ]:
# Some tests beforehand

results = model.predict("", task="classify")
print(results[0].probs.top1)  # Class index
print(results[0].names)       # List of all class
print(results[0].names[results[0].probs.top1])  # Check the results 


In [ ]:
import os
import csv
from ultralytics import YOLO

# Load the model
model = YOLO("/runs/classify/train3/weights/best.pt")  # path to the model

# Folder containing the images to classify
images_folder = ""

# Output CSV file
output_csv = ""

with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # CSV header
    writer.writerow(["image_name", "predicted_class"])

    # Iterate through all images in the folder
    for image_name in os.listdir(images_folder):
        # Filter by extension if necessary (.jpg, .png, etc.)
        if image_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(images_folder, image_name)

            # YOLOv8 inference in classification mode
            results = model.predict(image_path, task='classify')  

            # "results" is a list, each element corresponds to one image
            # results[0].probs contains a dict {class_index: probability}
            # results[0].names contains the index → class name mapping
            # In practice, you can simply do:
            #   predicted = results[0].probs.top1
            # to get the index of the most probable class 
            # or .names[.probs.top1] for the corresponding class name.

            # Example: get the most probable class
            r = results[0]
            top1_index = r.probs.top1  # index of the most probable class
            class_name = r.names[top1_index]  # name of the class

            # Write to the CSV
            writer.writerow([image_name, class_name])

print("Done! CSV output is", output_csv)


In [ ]:
#result control

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# load the CSV to check
csv_path = "v"
df = pd.read_csv(csv_path)

# check how many images were classified
print(f"Total number of classified images: {len(df)-1}")  # -1 to exclude the header

# count the occurrences of each class
class_counts = df["predicted_class"].value_counts()
print(class_counts)


